In [9]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ===========================
# DOWNLOAD REQUIRED DATA
# ===========================
nltk.download('punkt')
nltk.download('stopwords')

# ===========================
# PREPROCESS FUNCTION
# ===========================
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

# ===========================
# EVALUATION FUNCTION
# ===========================
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)

    print(f"\n{name} PERFORMANCE")
    print("="*50)
    print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return y_pred

# ===========================
# MAIN PROGRAM
# ===========================
if __name__ == "__main__":

    # ===========================
    # IMPROVED DATASET (BALANCED)
    # ===========================
    data = [

# POSITIVE (15)
("I love this movie, it was amazing!", "positive"),
("Great acting and fantastic story", "positive"),
("Best movie ever", "positive"),
("Awesome experience", "positive"),
("Fantastic direction and screenplay", "positive"),
("Loved every moment of the movie", "positive"),
("Superb acting and visuals", "positive"),
("Wonderful film, highly recommended", "positive"),
("Brilliant performance by actors", "positive"),
("Excellent storyline and execution", "positive"),
("Really enjoyed the movie", "positive"),
("One of the best films I have seen", "positive"),
("Amazing cinematography", "positive"),
("Great music and direction", "positive"),
("Absolutely loved the movie", "positive"),

# NEGATIVE (15)
("Worst movie ever", "negative"),
("Very boring and slow", "negative"),
("Waste of time", "negative"),
("Terrible acting", "negative"),
("Poor storyline and bad acting", "negative"),
("Disappointing experience", "negative"),
("Not worth watching", "negative"),
("Awful movie", "negative"),
("Horrible direction", "negative"),
("Bad screenplay and acting", "negative"),
("Completely boring film", "negative"),
("I did not like the movie", "negative"),
("Very disappointing movie", "negative"),
("Worst experience ever", "negative"),
("Not good at all", "negative"),

# NEUTRAL (15)
("It was okay", "neutral"),
("Average movie", "neutral"),
("Not bad not great", "neutral"),
("Decent film", "neutral"),
("It was fine", "neutral"),
("Nothing special about the movie", "neutral"),
("Mediocre experience", "neutral"),
("Just an average film", "neutral"),
("It was alright", "neutral"),
("Some parts were good", "neutral"),
("Okay movie overall", "neutral"),
("Neither good nor bad", "neutral"),
("Average acting and story", "neutral"),
("Not impressive but okay", "neutral"),
("It was a normal movie", "neutral"),
]

    texts = [d[0] for d in data]
    labels = [d[1] for d in data]

    print("\nDATASET INFO")
    print("="*50)
    print(f"Total Samples: {len(texts)}")
    print(f"Classes: {set(labels)}")

    # ===========================
    # PREPROCESSING
    # ===========================
    processed_texts = [preprocess_text(t) for t in texts]

    print("\nSample Preprocessing:")
    print("Before:", texts[0])
    print("After :", processed_texts[0])

    # ===========================
    # FEATURE EXTRACTION (IMPROVED)
    # ===========================
    print("\nFEATURE EXTRACTION (TF-IDF)")
    print("="*50)

    vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2), min_df=1)
    X = vectorizer.fit_transform(processed_texts)

    print("Feature Matrix Shape:", X.shape)

    # ===========================
    # TRAIN TEST SPLIT
    # ===========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=0.3, random_state=42, stratify=labels
    )

    print("\nTrain Size:", X_train.shape[0])
    print("Test Size :", X_test.shape[0])

    # ===========================
    # NAIVE BAYES
    # ===========================
    nb = MultinomialNB()
    nb.fit(X_train, y_train)
    nb_pred = evaluate_model(nb, X_test, y_test, "Naive Bayes")

    # ===========================
    # SVM (BALANCED FIX)
    # ===========================
    svm = LinearSVC(max_iter=2000, class_weight='balanced')
    svm.fit(X_train, y_train)
    svm_pred = evaluate_model(svm, X_test, y_test, "SVM")

    # ===========================
    # MODEL COMPARISON
    # ===========================
    nb_acc = accuracy_score(y_test, nb_pred)
    svm_acc = accuracy_score(y_test, svm_pred)

    print("\nMODEL COMPARISON")
    print("="*50)
    print(f"Naive Bayes Accuracy: {nb_acc*100:.2f}%")
    print(f"SVM Accuracy        : {svm_acc*100:.2f}%")

    best_model = svm if svm_acc >= nb_acc else nb

    # ===========================
    # NEW TEXT PREDICTION
    # ===========================
    print("\nNEW TEXT PREDICTION")
    print("="*50)

    test_sentences = [
        "Amazing movie with great acting",
        "Worst film ever",
        "It was okay not bad",
        "Very boring and slow"
    ]

    for sent in test_sentences:
        processed = preprocess_text(sent)
        vector = vectorizer.transform([processed])
        pred = best_model.predict(vector)[0]

        print(f"\nText: {sent}")
        print(f"Prediction: {pred.upper()}")

    # ===========================
    # CONFUSION MATRIX
    # ===========================
    print("\nCONFUSION MATRIX (SVM)")
    print("="*50)

    cm = confusion_matrix(y_test, svm_pred, labels=['positive','neutral','negative'])

    print("Rows = Actual, Columns = Predicted\n")
    print(cm)

    # ===========================
    # FEATURE IMPORTANCE (NB)
    # ===========================
    print("\nTOP WORDS (Naive Bayes)")
    print("="*50)

    feature_names = vectorizer.get_feature_names_out()

    for i, label in enumerate(nb.classes_):
        top = np.argsort(nb.feature_log_prob_[i])[-5:]
        words = [feature_names[j] for j in top]

        print(f"{label}: {', '.join(words)}")


DATASET INFO
Total Samples: 45
Classes: {'positive', 'neutral', 'negative'}

Sample Preprocessing:
Before: I love this movie, it was amazing!
After : love movie amazing

FEATURE EXTRACTION (TF-IDF)
Feature Matrix Shape: (45, 132)

Train Size: 31
Test Size : 14

Naive Bayes PERFORMANCE
Accuracy: 42.86%

Classification Report:
              precision    recall  f1-score   support

    negative       0.33      1.00      0.50         4
     neutral       1.00      0.40      0.57         5
    positive       0.00      0.00      0.00         5

    accuracy                           0.43        14
   macro avg       0.44      0.47      0.36        14
weighted avg       0.45      0.43      0.35        14


SVM PERFORMANCE
Accuracy: 50.00%

Classification Report:
              precision    recall  f1-score   support

    negative       0.36      1.00      0.53         4
     neutral       1.00      0.40      0.57         5
    positive       1.00      0.20      0.33         5

    accuracy   

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
